In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import gc

In [4]:
files = [
    "merged_1star.csv",
    "merged_2star.csv",
    "merged_3star.csv",
    "merged_4star.csv",
    "merged_5star.csv"
]

In [5]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# ✅ DEFINE FEATURES FIRST
features = ['X_Acc','Y_Acc','Z_Acc','X_Gyro','Y_Gyro','Z_Gyro']
SEQ_LEN = 50

scaler = StandardScaler()

for file in files:
    df = pd.read_csv(file)

    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

    df = df.dropna(subset=['Rating'])

    # ✅ NOW THIS WILL WORK
    scaler.partial_fit(df[features])

In [6]:
X = []
y = []

for file in files:
    print("Processing:", file)

    df = pd.read_csv(file)

    # clean columns
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
    df.columns = df.columns.str.strip()

    # 🔥 fix Sr.No issue
    if 'Sr.No.' in df.columns:
        df = df.rename(columns={'Sr.No.': 'SrNo'})

    df = df.drop(columns=['SrNo'], errors='ignore')

    # 🔥 remove NaN ratings
    df = df.dropna(subset=['Rating'])

    # 🔥 fix timestamp
    df['Timestamp'] = pd.to_numeric(df['Timestamp'], errors='coerce')

    # remove bad timestamps
    df = df.dropna(subset=['Timestamp'])

    # sort
    df = df.sort_values(by='Timestamp')

    # 🔥 detect sessions
    df['time_diff'] = df['Timestamp'].diff()
    df['session_id'] = (df['time_diff'] > 2000).cumsum()

    # drop timestamp columns
    df = df.drop(columns=['Timestamp', 'time_diff'])

    # normalize
    df[features] = scaler.transform(df[features])

    # 🔥 sequence creation per session
    for session in df['session_id'].unique():
        session_df = df[df['session_id'] == session]

        data = session_df[features].values
        labels = session_df['Rating'].values

        if len(data) < SEQ_LEN:
            continue

        for i in range(len(data) - SEQ_LEN):

            # 🔥 skip NaN labels
            if np.isnan(labels[i+SEQ_LEN]):
                continue

            X.append(data[i:i+SEQ_LEN])
            y.append(labels[i+SEQ_LEN])

    del df
    gc.collect()

Processing: merged_1star.csv
Processing: merged_2star.csv
Processing: merged_3star.csv
Processing: merged_4star.csv
Processing: merged_5star.csv


In [7]:
X = np.array(X, dtype=np.float32)
y = np.array(y)

# 🔥 remove any remaining NaNs
mask = ~np.isnan(y)

X = X[mask]
y = y[mask]

y = y.astype(np.int32)

y = y - 1   # 🔥 convert labels 1–5 → 0–4

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2347358, 50, 6)
y shape: (2347358,)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # 🔥 VERY IMPORTANT
)

In [9]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Dropout, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model

In [10]:
def transformer_block(inputs, head_size, num_heads, ff_dim, dropout=0.1):

    # Self-attention
    x = MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout
    )(inputs, inputs)

    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x + inputs)

    # Feed-forward
    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)

    x = Dropout(dropout)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + inputs)

In [11]:
def build_transformer(input_shape):

    inputs = Input(shape=input_shape)

    x = transformer_block(inputs, head_size=16, num_heads=1, ff_dim=32)
    x = transformer_block(x, head_size=16, num_heads=1, ff_dim=32)

    x = GlobalAveragePooling1D()(x)

    x = Dense(32, activation="relu")(x)
    x = Dropout(0.2)(x)

    outputs = Dense(5, activation="softmax")(x)

    model = Model(inputs, outputs)

    return model

In [12]:
transformer_model = build_transformer((SEQ_LEN, len(features)))

In [13]:
transformer_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [14]:
history_tf = transformer_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=12,
    validation_data=(X_test, y_test)
)

Epoch 1/10
156491/156491 ━━━━━━━━━━━━━━━━━━━━ 713s 4ms/step - accuracy: 0.6838 - loss: 0.7972 - val_accuracy: 0.7256 - val_loss: 0.6931
Epoch 2/10
156491/156491 ━━━━━━━━━━━━━━━━━━━━ 733s 4ms/step - accuracy: 0.7191 - loss: 0.7071 - val_accuracy: 0.7309 - val_loss: 0.6871
Epoch 3/10
156491/156491 ━━━━━━━━━━━━━━━━━━━━ 702s 4ms/step - accuracy: 0.7263 - loss: 0.6864 - val_accuracy: 0.7253 - val_loss: 0.7092
Epoch 4/10
156491/156491 ━━━━━━━━━━━━━━━━━━━━ 692s 4ms/step - accuracy: 0.7317 - loss: 0.6736 - val_accuracy: 0.7498 - val_loss: 0.6277
Epoch 5/10
156491/156491 ━━━━━━━━━━━━━━━━━━━━ 688s 4ms/step - accuracy: 0.7363 - loss: 0.6620 - val_accuracy: 0.7476 - val_loss: 0.6490
Epoch 6/10
156491/156491 ━━━━━━━━━━━━━━━━━━━━ 684s 4ms/step - accuracy: 0.7388 - loss: 0.6553 - val_accuracy: 0.7378 - val_loss: 0.6807
Epoch 7/10
156491/156491 ━━━━━━━━━━━━━━━━━━━━ 689s 4ms/step - accuracy: 0.7408 - loss: 0.6510 - val_accuracy: 0.7441 - val_loss: 0.6807
Epoch 8/10
156491/156491 ━━━━━━━━━━━━━━━━━━━━ 68

In [15]:
import numpy as np
from sklearn.metrics import classification_report

y_pred_tf = np.argmax(transformer_model.predict(X_test), axis=1)

print(classification_report(y_test, y_pred_tf))

14671/14671 ━━━━━━━━━━━━━━━━━━━━ 34s 2ms/step
              precision    recall  f1-score   support

           0       0.67      0.85      0.75    114900
           1       0.76      0.66      0.70     97116
           2       0.75      0.74      0.74    141047
           3       0.80      0.58      0.67     30400
           4       0.85      0.76      0.80     86009

    accuracy                           0.74    469472
   macro avg       0.76      0.72      0.73    469472
weighted avg       0.75      0.74      0.74    469472



In [16]:
transformer_model.save("transformer_exp_model.keras")